# 📘 Notebook 10: Better Deep Q-Networks

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module C: Deep Value-Based Learning · Notebook 3 of 4 in this module · (10 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Demonstrate the **maximisation bias** of Q-learning and explain where it comes from
- Implement **Double DQN** and say which network chooses and which one evaluates
- Describe the **dueling** architecture and its split into state value and advantages
- Explain **prioritised experience replay** and why its uneven sampling has to be corrected
- Recognise when uniform random **exploration** is the wrong choice, and replace it
- Train an agent that plays **Flappy Bird**, and judge the improvements honestly against the noise between runs

> **Prerequisites:** Notebook 09: the DQN algorithm, its replay buffer and its target network.
>
> 🔭 **Why this notebook matters later:** the three refinements of this notebook are standard parts of modern value-based agents. The project at the end also teaches something no formula does: on a new game, how the agent **explores** can matter more than which algorithm it runs.

> 📦 **Libraries used in this notebook.**
>
> - **flappy-bird-gymnasium** is an open-source Gymnasium environment for the game Flappy Bird.
> - **PyTorch**, **Gymnasium** and **imageio**, as before.
>
> No GPU is needed. The small experiments take seconds. **Training the bird takes several minutes**, and the notebook says when.

In [ ]:
%pip install -q "gymnasium[classic-control]>=1.3" flappy-bird-gymnasium imageio imageio-ffmpeg torch

In [ ]:
import base64
import copy
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
import flappy_bird_gymnasium
import torch
import torch.nn as nn
from IPython.display import HTML, display

torch.set_num_threads(2)


def show_video(frames, fps=30, width=240):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. Three weaknesses

Notebook 09 ended with a list of things DQN does badly. This notebook takes up three of them:

| Weakness | Refinement | Section |
|---|---|---|
| The `max` in the target makes the values too optimistic | **Double DQN** | 2 and 3 |
| The network must learn the value of every action separately, even where the action hardly matters | **Dueling networks** | 4 |
| All stored transitions are replayed equally often | **Prioritised replay** | 5 |

Each was published as its own paper between 2015 and 2016. In 2017 a DeepMind paper combined these and three further ideas into one agent, called *Rainbow*, and showed that the improvements add up.

## 2. The maximisation bias

### Intuition first
Ten friends each guess the weight of the same cake. Their guesses scatter around the true weight, some too high and some too low, and their **average** is a good estimate. Now take the **largest** of the ten guesses. It is almost certainly too high: taking a maximum picks out whoever made the biggest error upwards.

The Q-learning target contains exactly this operation. $\max_a Q(S', a)$ is the largest of several estimates. If the estimates are noisy, and during learning they always are, the maximum is systematically **too large**, even when each estimate on its own is unbiased.

In [ ]:
rng = np.random.default_rng(0)

print("Every action has a true value of 0. The estimates are noisy (standard deviation 1).")
for n_actions in (2, 4, 10, 50):
    estimates = rng.normal(0, 1, (100000, n_actions))
    print(f"  {n_actions:>2} actions: the largest estimate is on average {estimates.max(axis=1).mean():.2f}")

With all true values equal to zero, the correct answer is 0. The maximum of noisy estimates is far from it, and the more actions there are, the worse it gets. Because targets feed into other targets through bootstrapping, the error does not stay where it was made.

### A cure: ask a second opinion
Use **two independent** sets of estimates. Let the first one **choose** the action that looks best, and let the second one say what that action is **worth**. The first estimator will still tend to choose an action whose value it has overestimated. But the second estimator's error for that action is unrelated, so it is as likely to be too low as too high.

In [ ]:
print("Choose the action with one set of estimates, evaluate it with a second, independent set:")
for n_actions in (2, 4, 10, 50):
    first = rng.normal(0, 1, (100000, n_actions))
    second = rng.normal(0, 1, (100000, n_actions))
    chosen = first.argmax(axis=1)
    print(f"  {n_actions:>2} actions: value of the chosen action on average "
          f"{second[np.arange(100000), chosen].mean():+.3f}")

The bias is gone.

### Does it matter for behaviour?
A classic small example from Sutton and Barto's textbook shows that it does. From a start state A the agent can go **right**, which ends the episode with reward 0, or **left** to a state B. In B there are ten actions, all of which end the episode with a random reward of −0.1 on average. So going left loses 0.1 on average and **right is the better choice**. An agent with epsilon 0.1 that has learned this will still go left on 5% of the episodes, by exploration.

But some of the ten actions in B will look good by chance, the `max` reports the luckiest of them, and Q-learning concludes that B is worth visiting. **Double Q-learning** keeps two tables and uses one to choose and the other to evaluate.

In [ ]:
def bias_experiment(double, n_b=10, runs=500, episodes=300, alpha=0.1, epsilon=0.1, seed=0):
    """Percentage of runs that go LEFT (the worse action) in each episode. State B has n_b actions."""
    rng = np.random.default_rng(seed)
    went_left = np.zeros(episodes)

    def pick(values):
        if rng.random() < epsilon:
            return int(rng.integers(len(values)))
        return int(rng.choice(np.flatnonzero(values == values.max())))

    for run in range(runs):
        QA = np.zeros((2, 2))                       # state A: two tables x (left, right)
        QB = np.zeros((2, n_b))                     # state B: two tables x its actions
        for episode in range(episodes):
            k = int(rng.integers(2)) if double else 0            # which table is updated this time
            action = pick(QA.sum(axis=0) if double else QA[0])
            if action == 1:                                       # right: the episode ends with reward 0
                QA[k, 1] += alpha * (0.0 - QA[k, 1])
                continue
            went_left[episode] += 1
            if double:
                future = QB[1 - k, QB[k].argmax()]                # table k chooses, the other evaluates
            else:
                future = QB[0].max()                              # the same table chooses and evaluates
            QA[k, 0] += alpha * (0.0 + future - QA[k, 0])
            b_action = pick(QB.sum(axis=0) if double else QB[0])
            reward = rng.normal(-0.1, 1.0)
            k = int(rng.integers(2)) if double else 0
            QB[k, b_action] += alpha * (reward - QB[k, b_action])
    return 100 * went_left / runs


left_q = bias_experiment(double=False)
left_double = bias_experiment(double=True)

plt.figure(figsize=(7, 4))
plt.plot(left_q, label="Q-learning")
plt.plot(left_double, label="Double Q-learning")
plt.axhline(5, color="gray", linestyle=":", label="best possible (5%)")
plt.title("How often the worse action is chosen")
plt.xlabel("episode"); plt.ylabel("% of 500 runs that went left"); plt.grid(True); plt.legend()
plt.show()

for episode in (25, 100, 300):
    print(f"episode {episode:>3}: Q-learning goes left in {left_q[episode - 1]:3.0f}% of runs, "
          f"Double Q-learning in {left_double[episode - 1]:3.0f}%")

Early on, Q-learning takes the **worse** action in the great majority of runs, because of nothing but the `max`. It recovers slowly as its estimates improve. Double Q-learning is misled far less and settles sooner.

## 3. Double DQN

### Formal definition
DQN already has two networks, the online network and the target network. Hado van Hasselt and colleagues (2016) noticed that they can play the two roles, with no third network needed:

$$\text{DQN:}\quad R + \gamma\,\max_a \hat{q}(S', a;\mathbf{w}^-) \qquad\qquad \text{Double DQN:}\quad R + \gamma\,\hat{q}\big(S', \arg\max_a \hat{q}(S', a;\mathbf{w});\ \mathbf{w}^-\big)$$

The **online** network chooses the best next action. The **target** network evaluates it. The two are not fully independent, since one is a delayed copy of the other, but they differ enough to remove much of the bias. In code it is two lines:

```python
best = online(next_states).argmax(dim=1, keepdim=True)        # online network chooses
future = target(next_states).gather(1, best).squeeze(1)       # target network evaluates
```

## 4. Dueling networks

### Intuition first
When the road ahead is empty, it makes almost no difference whether a driver steers a hair to the left or to the right. What matters is the situation: is this a good place to be or not? Only when an obstacle appears does the choice of action become important.

An ordinary Q-network has to learn the full value of every action separately, even in the many states where all actions are worth nearly the same.

### Formal definition
The **dueling architecture** (Ziyu Wang and colleagues, 2016) splits the network into two heads after the shared layers:

- a **value** head with one output, $V(s)$: how good is the state?
- an **advantage** head with one output per action, $A(s, a)$: how much better or worse than average is each action here?

They are combined into action values by

$$Q(s, a) = V(s) + A(s, a) - \frac{1}{|\mathcal{A}|}\sum_{a'} A(s, a')$$

Subtracting the mean advantage makes the split unique. Without it one could add any number to $V$ and subtract it from every $A$ without changing $Q$, and the two heads would have no fixed meaning.

### Why it matters
Every training sample, whatever action it contains, improves $V(s)$, and $V(s)$ is shared by all actions. The network learns which states are valuable without having to try every action in each of them.

In [ ]:
class DuelingNetwork(nn.Module):
    def __init__(self, state_size, n_actions, hidden=128):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(state_size, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.value = nn.Linear(hidden, 1)                 # how good is the state?
        self.advantage = nn.Linear(hidden, n_actions)     # how much better or worse is each action?

    def forward(self, state):
        features = self.body(state)
        advantage = self.advantage(features)
        return self.value(features) + advantage - advantage.mean(dim=-1, keepdim=True)


torch.manual_seed(0)
network = DuelingNetwork(4, 2)
state = torch.tensor([0.0, 0.1, 0.02, -0.1])

with torch.no_grad():
    features = network.body(state)
    V, A = network.value(features), network.advantage(features)
    Q = network(state)

print("V(s):          ", V.numpy().round(3))
print("A(s, a):       ", A.numpy().round(3))
print("Q(s, a):       ", Q.numpy().round(3))
print("mean of Q(s, a):", round(Q.mean().item(), 3), "(equal to V, because the advantages were centred)")

From the outside a dueling network is used exactly like the plain one: a state goes in, one value per action comes out. Nothing else in the algorithm changes.

## 5. Prioritised replay

### Intuition first
A student preparing for an examination does not reread every page equally often. They spend their time on the exercises they got **wrong**. Uniform replay rereads everything equally, including the thousands of uneventful transitions the network already predicts well.

### Formal definition
In **prioritised experience replay** (Tom Schaul and colleagues, 2016) each stored transition has a **priority**, the size of its last TD error $|\delta|$. Transition $i$ is sampled with probability

$$P(i) = \frac{p_i^{\,\alpha}}{\sum_k p_k^{\,\alpha}} \qquad \text{with } p_i = |\delta_i| + \text{a small constant}$$

The exponent $\alpha$ sets how strongly the priorities count: 0 gives uniform sampling.

Sampling some transitions more often than others changes what the network is trained on, and would bias the result. Each sample's loss is therefore multiplied by an **importance weight** $w_i = (N \cdot P(i))^{-\beta}$, which is small for transitions that are drawn often.

The class below extends the replay buffer of Notebook 09. (The uniform buffer now also returns the positions of its samples and a weight of 1 for each, so that the two kinds can be used in the same way.)

In [ ]:
class ReplayBuffer:
    """Uniform replay, as in Notebook 09. sample() also returns the positions and a weight of 1 per sample."""

    def __init__(self, capacity, state_size):
        self.states = np.zeros((capacity, state_size), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_size), dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.capacity, self.size, self.position = capacity, 0, 0

    def add(self, state, action, reward, next_state, terminated):
        i = self.position
        self.states[i], self.actions[i], self.rewards[i] = state, action, reward
        self.next_states[i], self.terminated[i] = next_state, float(terminated)
        self.position = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def batch(self, index):
        return tuple(torch.as_tensor(array[index]) for array in
                     (self.states, self.actions, self.rewards, self.next_states, self.terminated))

    def sample(self, batch_size, rng):
        index = rng.integers(0, self.size, batch_size)
        return self.batch(index), index, torch.ones(batch_size)

    def update(self, index, errors):
        pass                                             # uniform replay ignores the errors


class PrioritisedBuffer(ReplayBuffer):
    """Transitions with a large TD error are sampled more often."""

    def __init__(self, capacity, state_size, alpha=0.6, beta=0.4):
        super().__init__(capacity, state_size)
        self.priorities = np.zeros(capacity, dtype=np.float64)
        self.alpha, self.beta = alpha, beta

    def add(self, state, action, reward, next_state, terminated):
        # a new transition gets the highest priority so far, so that it is replayed at least once
        self.priorities[self.position] = self.priorities[:self.size].max() if self.size else 1.0
        super().add(state, action, reward, next_state, terminated)

    def sample(self, batch_size, rng):
        probabilities = self.priorities[:self.size] ** self.alpha
        probabilities /= probabilities.sum()
        index = rng.choice(self.size, batch_size, p=probabilities)
        weights = (self.size * probabilities[index]) ** (-self.beta)     # correct for the uneven sampling
        weights /= weights.max()
        return self.batch(index), index, torch.as_tensor(weights, dtype=torch.float32)

    def update(self, index, errors):
        self.priorities[index] = np.abs(errors) + 0.01

In [ ]:
rng = np.random.default_rng(0)
demo = PrioritisedBuffer(capacity=5, state_size=1)
for i in range(5):
    demo.add([float(i)], 0, 0.0, [float(i)], False)

demo.update(np.arange(5), errors=np.array([0.1, 0.1, 0.1, 0.1, 5.0]))     # transition 4 was predicted badly

counts = np.zeros(5, dtype=int)
for draw in range(2000):
    batch, index, weights = demo.sample(1, rng)
    counts[index[0]] += 1

print("TD errors:             0.1   0.1   0.1   0.1   5.0")
print("times sampled of 2000:", counts)

The badly predicted transition is drawn many times more often than the others. As soon as the network has learned from it, its error shrinks, its priority falls, and attention moves elsewhere.

> ⚠️ **Common pitfalls**
>
> - **Double DQN with the roles swapped.** The online network chooses, the target network evaluates.
> - **A dueling network without the mean subtracted.** It trains, but $V$ and $A$ drift and learning becomes less stable.
> - **Prioritised replay without importance weights.** The agent then over-learns the rare, surprising transitions.
> - **Expecting every refinement to help on every problem.** Exercise 3 checks them on CartPole.

## 6. One function for all variants

The DQN function of Notebook 09, extended with four switches: `double`, `dueling`, `prioritised`, and `explore_probabilities`, which Section 7 will explain.

In [ ]:
def plain_network(state_size, n_actions, hidden=128):
    return nn.Sequential(nn.Linear(state_size, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                         nn.Linear(hidden, n_actions))


def dqn(env, steps, double=False, dueling=False, prioritised=False, explore_probabilities=None,
        hidden=128, learning_rate=5e-4, gamma=0.99, batch_size=64, buffer_size=100000, warm_up=1000,
        train_every=4, tau=0.005, explore_steps=20000, epsilon_end=0.02, seed=0, log=None, score_key=None):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    state_size, n_actions = env.observation_space.shape[0], env.action_space.n
    online = (DuelingNetwork if dueling else plain_network)(state_size, n_actions, hidden)
    target = copy.deepcopy(online)
    optimiser = torch.optim.Adam(online.parameters(), lr=learning_rate)
    buffer = (PrioritisedBuffer if prioritised else ReplayBuffer)(buffer_size, state_size)
    history, episode_return = [], 0.0
    state, info = env.reset(seed=seed)
    for step in range(1, steps + 1):
        epsilon = max(epsilon_end, 1 - step / explore_steps)
        if rng.random() < epsilon:
            action = int(rng.choice(n_actions, p=explore_probabilities))
        else:
            with torch.no_grad():
                action = int(online(torch.as_tensor(state, dtype=torch.float32)).argmax())
        next_state, reward, terminated, truncated, info = env.step(action)
        buffer.add(state, action, reward, next_state, terminated)
        state, episode_return = next_state, episode_return + reward
        if terminated or truncated:
            history.append(info[score_key] if score_key else episode_return)
            episode_return = 0.0
            state, info = env.reset()
            if log and len(history) % log == 0:
                print(f"step {step:>7,}   episode {len(history):>5}   average of the last {log} episodes: {np.mean(history[-log:]):7.2f}")
        if buffer.size >= warm_up and step % train_every == 0:
            (states, actions, rewards, next_states, done), index, weights = buffer.sample(batch_size, rng)
            with torch.no_grad():
                if double:
                    best = online(next_states).argmax(dim=1, keepdim=True)          # online network chooses
                    future = target(next_states).gather(1, best).squeeze(1)         # target network evaluates
                else:
                    future = target(next_states).max(dim=1).values
                targets = rewards + gamma * (1 - done) * future
            predicted = online(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            errors = targets - predicted
            loss = (weights * nn.functional.smooth_l1_loss(predicted, targets, reduction="none")).mean()
            optimiser.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(online.parameters(), 10.0)
            optimiser.step()
            buffer.update(index, errors.detach().numpy())
            with torch.no_grad():
                for w, w_target in zip(online.parameters(), target.parameters()):
                    w_target.mul_(1 - tau).add_(tau * w)
    return online, np.array(history)

## 7. Project: Flappy Bird

### The game
A bird flies to the right at constant speed and falls under gravity. One button makes it flap upwards. Pairs of pipes with a gap between them come towards it, and touching a pipe or the ground ends the game. The score is the number of pipes passed.

| | |
|---|---|
| **State** | 12 numbers: the horizontal position and the top and bottom of the gap for the last pipe and the next two pipes, and the bird's height, vertical speed and rotation |
| **Actions** | 0 do nothing, 1 flap |
| **Reward** | +0.1 for every frame survived, +1 for each pipe passed, −1 for dying, −0.5 for touching the top of the screen |

The game runs at 30 frames per second, and the agent decides on every frame.

In [ ]:
flappy = gym.make("FlappyBird-v0", use_lidar=False, render_mode="rgb_array")
state, info = flappy.reset(seed=0)
print("observation:", np.round(state, 2))
print("actions:    ", flappy.action_space.n)


def play_flappy(env, policy, seed=0, max_frames=3000, record=False):
    """One game. Returns the score (pipes passed), the frames survived, and the recorded pictures."""
    state, info = env.reset(seed=seed)
    frames, survived = [], 0
    terminated = truncated = False
    while not (terminated or truncated) and survived < max_frames:
        if record:
            frames.append(env.render())
        state, reward, terminated, truncated, info = env.step(policy(state))
        survived += 1
    return info["score"], survived, frames


uniform = np.random.default_rng(0)
score, survived, frames = play_flappy(flappy, lambda state: int(uniform.integers(2)), record=True)
print(f"random agent: score {score}, survived {survived} frames")
show_video(frames)

### Why the random agent teaches nothing

Watch the video. The bird shoots to the top of the screen and stays there until it meets the first pipe. A "random" agent flaps on half of all frames, which is far too often: one flap carries the bird upwards for many frames. This matters for learning. Epsilon-greedy exploration with uniformly random actions will produce exactly this behaviour during the early, exploratory phase, so the replay buffer fills with transitions from the ceiling and almost none from the height where the gaps are.

Exploration does not have to be uniform. Let the exploring agent flap on only **one frame in ten**:

In [ ]:
plain_env = gym.make("FlappyBird-v0", use_lidar=False)


def exploration_statistics(flap_probability, games=200):
    chooser = np.random.default_rng(0)
    heights, frames_survived = [], []
    for game in range(games):
        state, info = plain_env.reset(seed=game)
        terminated = truncated = False
        frames = 0
        while not (terminated or truncated):
            state, reward, terminated, truncated, info = plain_env.step(int(chooser.random() < flap_probability))
            heights.append(state[9])                   # the bird's height: 0 is the top of the screen
            frames += 1
        frames_survived.append(frames)
    heights = np.array(heights)
    return np.mean(frames_survived), 100 * np.mean(heights < 0.05)


for probability in (0.5, 0.1):
    survived, at_ceiling = exploration_statistics(probability)
    print(f"flapping on {100 * probability:.0f}% of frames: survives {survived:5.1f} frames on average, "
          f"spends {at_ceiling:4.1f}% of them at the very top of the screen")

With fewer flaps the bird spends its time in the part of the screen where the game is actually played, and the experience it collects is about pipes and gaps. This is what the `explore_probabilities` switch of `dqn` is for: `[0.9, 0.1]` means that an exploratory action is "do nothing" with probability 0.9 and "flap" with probability 0.1.

### Training
We use Double DQN with a dueling network and the adjusted exploration. Epsilon falls from 1 to 0.001 over the first 30,000 steps. It has to end very low: in this game a single unnecessary flap can be fatal.

**This cell takes several minutes.** It prints the average score, in pipes passed, of every 500 games. Expect it to stay near zero for a long time and then to rise.

In [ ]:
bird, scores = dqn(plain_env, 300000, double=True, dueling=True, explore_probabilities=[0.9, 0.1],
                   explore_steps=30000, epsilon_end=0.001, score_key="score", log=500)

In [ ]:
def smooth(values, window=100):
    return np.convolve(values, np.ones(window) / window, mode="valid")


plt.figure(figsize=(8, 4))
plt.plot(smooth(scores))
plt.title("Flappy Bird: pipes passed per game during training")
plt.xlabel("game"); plt.ylabel("score (average of 100 games)"); plt.grid(True)
plt.show()


def greedy(network):
    def policy(state):
        with torch.no_grad():
            return int(network(torch.as_tensor(state, dtype=torch.float32)).argmax())
    return policy


results = [play_flappy(plain_env, greedy(bird), seed=1000 + game)[0] for game in range(30)]
print(f"trained bird over 30 games: average score {np.mean(results):.1f}, median {np.median(results):.0f}, "
      f"best {max(results)}, worst {min(results)}")

The random agent passed no pipes at all. The trained bird passes many, although its games differ a great deal from one another: read the best and the worst score. A single late flap ends a game, so one small weakness in the learned values costs everything that would have followed.

In [ ]:
score, survived, frames = play_flappy(flappy, greedy(bird), seed=1000 + int(np.argmax(results)), max_frames=900, record=True)
print(f"the first {survived} frames of the bird's best game (score so far: {score})")
show_video(frames[::2], fps=15)

## 8. What actually helped?

It is tempting to credit the result to the refinements of Sections 3 to 5. Honesty requires a comparison, and comparisons in reinforcement learning are expensive: each variant must be trained several times, because two runs of the **same** variant can differ by more than two different variants do.

We ran that comparison while preparing this notebook, with one to three training runs of 300,000 steps per variant, and evaluated each final network over 30 games:

| Exploration | Double | Dueling | Average score of each run |
|---|---|---|---|
| uniform (flap 50%) | no | no | 1.4 (one run) |
| uniform (flap 50%) | yes | yes | 7.4, 29.9 |
| flap 10% | no | no | 12.1, 22.6, 15.4 |
| flap 10% | yes | no | 2.4, 10.4 |
| flap 10% | yes | yes | 37.1, 28.5, 21.3 |

Read the table with the spread in mind. Changing the **exploration** helped clearly: every run with adjusted exploration beat the plain agent with uniform exploration. Beyond that the picture is not tidy. Double and dueling together gave the best row, yet Double DQN on its own did worse than plain DQN in our two runs, and the runs of one row overlap with those of another. Two or three runs are too few to rank the variants. These numbers come from our own runs. Yours will differ.

The lesson is general. A refinement published with results on fifty Atari games is not guaranteed to pay off on your problem, and a change that needs no new algorithm at all, such as exploring in a way that suits the game, can be worth more.

---
## ✏️ Exercises

The exercises use `dqn`, `DuelingNetwork`, `PrioritisedBuffer`, `bias_experiment`, `play_flappy`, `greedy` and the trained `bird`.

### Exercise 1 (More actions, more bias)
In the experiment of Section 2, state B had ten actions. `bias_experiment` takes the number of actions in B as the parameter `n_b`. Run Q-learning (`double=False`) for 50 episodes with 2, 10 and 50 actions in B, and print the percentage of runs that go left in the last episode. Do the same for Double Q-learning. Explain the trend using the first table of Section 2.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for n_b in (2, 10, 50):
    q = bias_experiment(double=False, n_b=n_b, episodes=50)[-1]
    d = bias_experiment(double=True, n_b=n_b, episodes=50)[-1]
    print(f"{n_b:>2} actions in B: at episode 50, Q-learning goes left in {q:3.0f}% of runs, Double Q-learning in {d:3.0f}%")
```

*The more actions there are in B, the larger the luckiest of their noisy estimates, exactly as in the first table of Section 2. A larger maximum makes B look more attractive, and Q-learning is misled for longer. Double Q-learning is much less affected. Games with many actions suffer most from the maximisation bias.*
</details>

### Exercise 2 (Uniform exploration on Flappy Bird)
Train a second bird with the same settings as in Section 7 but with ordinary uniform exploration (leave out `explore_probabilities`). **This takes as long as the training in the notebook.** Evaluate it over 30 games and compare with the first bird. Does your result agree with the table in Section 8?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
uniform_bird, uniform_scores = dqn(plain_env, 300000, double=True, dueling=True,
                                   explore_steps=30000, epsilon_end=0.001, score_key="score", log=1000)

uniform_results = [play_flappy(plain_env, greedy(uniform_bird), seed=1000 + game)[0] for game in range(30)]
print(f"uniform exploration: average score {np.mean(uniform_results):.1f}, best {max(uniform_results)}")
print(f"flap 10% (notebook): average score {np.mean(results):.1f}, best {max(results)}")
```

*In most runs the bird that explored uniformly is clearly behind, because its early experience came from the ceiling. But this is one run against one run. If your result disagrees with the table, that is not an error: it is the spread that Section 8 warns about.*
</details>

### Exercise 3 (The refinements on CartPole)
Do Double DQN, the dueling network and prioritised replay help on CartPole? Train for 20,000 steps with `learning_rate=2.5e-4, explore_steps=8000` (the settings of Notebook 09), three seeds each, for: plain DQN, `double=True`, `dueling=True`, and `prioritised=True` with `buffer_size=20000`. For each run print the average of the last 10 training episodes. Which differences are larger than the spread between seeds?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
cartpole = gym.make("CartPole-v1")
variants = {"plain DQN": {}, "double": dict(double=True), "dueling": dict(dueling=True),
            "prioritised": dict(prioritised=True, buffer_size=20000)}

for name, switches in variants.items():
    finals = []
    for seed in range(3):
        network, returns = dqn(cartpole, 20000, learning_rate=2.5e-4, explore_steps=8000, seed=seed, **switches)
        finals.append(returns[-10:].mean())
    print(f"{name:12s} last 10 episodes of three runs: " + "  ".join(f"{f:4.0f}" for f in finals))
```

*On CartPole the four variants land in the same range, and the differences between them are no larger than the differences between seeds of one variant. CartPole has two actions and dense rewards, so there is little maximisation bias to remove and few rare transitions to prioritise. The refinements were designed for harder problems. An honest experiment can end with "no measurable difference", and that is a result.*
</details>

### Exercise 4 (Inside the dueling network)
Take the trained `bird` (a `DuelingNetwork`). Play one game with it and record, for every frame, the state value $V(s)$ and the difference between the two advantages $A(s, \text{flap}) - A(s, \text{nothing})$. Plot both against the frame number for the first 300 frames. When is the difference between the actions large?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
state, info = plain_env.reset(seed=1000 + int(np.argmax(results)))
values, gaps = [], []
for frame in range(300):
    s = torch.as_tensor(state, dtype=torch.float32)
    with torch.no_grad():
        features = bird.body(s)
        values.append(bird.value(features).item())
        advantage = bird.advantage(features)
        gaps.append((advantage[1] - advantage[0]).item())
        action = int(bird(s).argmax())
    state, reward, terminated, truncated, info = plain_env.step(action)
    if terminated or truncated:
        break

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(values); axes[0].set_ylabel("V(s)"); axes[0].grid(True)
axes[0].set_title("The two heads of the dueling network during one game")
axes[1].plot(gaps); axes[1].axhline(0, color="gray", linestyle=":")
axes[1].set_ylabel("A(flap) - A(nothing)"); axes[1].set_xlabel("frame"); axes[1].grid(True)
plt.show()

print(f"frames on which flapping was preferred: {sum(g > 0 for g in gaps)} of {len(gaps)}")
```

*The lower curve is below zero for most frames: usually doing nothing is the better choice, and the bird flaps only on the frames where the curve rises above zero. That is why uniform exploration, with a flap on every second frame, suited this game so badly. The state value in the upper plot changes more slowly and follows how comfortable the bird's situation is.*
</details>

### Exercise 5 (Prioritised replay, looked at closely (a little harder))
Fill a `PrioritisedBuffer` of capacity 1,000 with transitions whose TD errors are drawn from `np.abs(rng.normal(0, 1, 1000))`. Draw 20,000 single samples for each of `alpha = 0`, `0.6` and `1` (set `buffer.alpha` before sampling) and report, for each, what share of all draws went to the 100 transitions with the largest errors. Then print the importance weight of the transition with the largest error and of the one with the smallest, for `alpha = 0.6`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
rng = np.random.default_rng(1)
buffer = PrioritisedBuffer(capacity=1000, state_size=1)
for i in range(1000):
    buffer.add([0.0], 0, 0.0, [0.0], False)
errors = np.abs(rng.normal(0, 1, 1000))
buffer.update(np.arange(1000), errors)
top = set(np.argsort(errors)[-100:])

for alpha in (0.0, 0.6, 1.0):
    buffer.alpha = alpha
    batch, index, weights = buffer.sample(20000, rng)
    share = np.mean([i in top for i in index])
    print(f"alpha = {alpha}: the 100 largest errors receive {100 * share:.0f}% of the draws")

buffer.alpha = 0.6
probabilities = buffer.priorities ** 0.6 / (buffer.priorities ** 0.6).sum()
weights = (1000 * probabilities) ** (-buffer.beta)
weights /= weights.max()
print(f"importance weight of the largest error:  {weights[errors.argmax()]:.2f}")
print(f"importance weight of the smallest error: {weights[errors.argmin()]:.2f}")
```

*With alpha = 0 every transition is equally likely, so 100 of 1,000 get 10% of the draws. As alpha grows, the transitions with large errors take a larger share. The importance weights run the other way: the transition that is sampled most often counts least in each update, which keeps the overall effect of each transition on the network in balance.*
</details>

## 🔑 Key takeaways

- The **max** over noisy estimates is biased upwards, and more actions make the bias larger.
- **Double DQN** lets the online network **choose** the next action and the target network **evaluate** it.
- A **dueling** network computes $Q = V + A - \text{mean}(A)$: one head for the state, one for the actions.
- **Prioritised replay** samples surprising transitions more often and corrects for it with **importance weights**.
- **Exploration should suit the game.** In Flappy Bird, flapping on half of all frames teaches the agent about the ceiling and nothing else.
- Improvements must be judged against the **spread between runs**. On small problems the refinements may make no measurable difference.

---
**Next:** *Notebook 11: Learning from Pixels*, where the agent is given the screen and nothing else.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*